<a href="https://colab.research.google.com/github/viettungvuong/Distillation-Playground/blob/main/Distillation_Playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 99.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


**Huggingface login**

In [1]:
from huggingface_hub import login

login()

**Downloading dataset**

In [2]:
from datasets import load_dataset

dataset = load_dataset("OpenHust/vietnamese-summarization")

In [3]:
from transformers import AutoProcessor, AutoModelForCausalLM

model_name = "google/gemma-4-E4B-it"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

**Data processing**

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Document', 'Summary', 'Dataset'],
        num_rows: 74564
    })
})

In [6]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)

train_set = split_dataset['train']
test_set = split_dataset['test']

In [7]:
train_set[:5]

{'Unnamed: 0': [1705, None, None, None, 8876],
 'Document': ['Hiện trường tai nạn .5h ngày 3/10 , xe bồn do tài xế Nguyễn Thành Công cầm lái , dừng ở làn khẩn cấp trên cao tốc TP HCM - Long Thành - Dầu Giây ( đoạn qua địa bàn huyện Long Thành , Đồng Nai ) .Một lúc sau xe cấp cứu mang biển số tỉnh Lâm Đồng chạy tốc độ cao , hướng từ Long Thành về TP HCM , lao tới đâm mạnh vào đuôi .Cú tông gây tiếng động lớn , phần đầu xe cấp cứu bẹp dúm , tài xế mắc kẹt trong cabin .Bệnh nhân ( bị gãy 2 chân được xe cấp cứu chở lên Sài Gòn chữa trị ) và 2 người nhà bị thương nặng , riêng bác sĩ đi trên xe chỉ bị choáng .Các nạn nhân được đưa đến Bệnh viện Long Thành nhưng tài xế xe cấp cứu đã tử vong .Bệnh nhân và một người nhà bị chấn thương sọ não tiếp tục được chuyển lên Bệnh viện Chợ Rẫy ( TP HCM ) .Vụ tai nạn không gây ảnh hưởng đến giao thông trên cao tốc , nguyên nhân đang được điều tra .Cú đâm khiến xe cấp cứu bị nát phần đầu .Được đầu tư hơn 20.600 tỷ đồng , dài 55 km qua địa phận TP HCM và tỉ

In [8]:
def normalize_text(row):
    # Only handle lowercase normalization
    row['Document'] = str(row['Document']).lower()
    row['Summary'] = str(row['Summary']).lower()
    return row

def add_special_tokens(row):
    # Create a new column combining Document and Summary with tokens
    row['Processed_Input'] = f"<bos> <|turn> {row['Document']} <turn|> <|turn> {row['Summary']} <turn|>"
    return row

# Apply normalization
train_set = train_set.map(normalize_text)
test_set = test_set.map(normalize_text)

# Apply special tokens and create new column
train_set = train_set.map(add_special_tokens)
test_set = test_set.map(add_special_tokens)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

In [9]:
# Verify normalization
display(train_set[0]['Processed_Input'])

'<bos> <|turn> hiện trường tai nạn .5h ngày 3/10 , xe bồn do tài xế nguyễn thành công cầm lái , dừng ở làn khẩn cấp trên cao tốc tp hcm - long thành - dầu giây ( đoạn qua địa bàn huyện long thành , đồng nai ) .một lúc sau xe cấp cứu mang biển số tỉnh lâm đồng chạy tốc độ cao , hướng từ long thành về tp hcm , lao tới đâm mạnh vào đuôi .cú tông gây tiếng động lớn , phần đầu xe cấp cứu bẹp dúm , tài xế mắc kẹt trong cabin .bệnh nhân ( bị gãy 2 chân được xe cấp cứu chở lên sài gòn chữa trị ) và 2 người nhà bị thương nặng , riêng bác sĩ đi trên xe chỉ bị choáng .các nạn nhân được đưa đến bệnh viện long thành nhưng tài xế xe cấp cứu đã tử vong .bệnh nhân và một người nhà bị chấn thương sọ não tiếp tục được chuyển lên bệnh viện chợ rẫy ( tp hcm ) .vụ tai nạn không gây ảnh hưởng đến giao thông trên cao tốc , nguyên nhân đang được điều tra .cú đâm khiến xe cấp cứu bị nát phần đầu .được đầu tư hơn 20.600 tỷ đồng , dài 55 km qua địa phận tp hcm và tỉnh đồng nai , cao tốc tp hcm - long thành - dầu

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
print("Special tokens: ", tokenizer.special_tokens_map)
sep_token_id = tokenizer.convert_tokens_to_ids("<|turn>")
print(f"Sep token is {sep_token_id}")

Special tokens:  {'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask>', 'audio_token': '<|audio|>', 'boa_token': '<|audio>', 'boi_token': '<|image>', 'eoa_token': '<audio|>', 'eoc_token': '<channel|>', 'eoi_token': '<image|>', 'eot_token': '<turn|>', 'escape_token': '<|"|>', 'etc_token': '<tool_call|>', 'etd_token': '<tool|>', 'etr_token': '<tool_response|>', 'image_token': '<|image|>', 'soc_token': '<|channel>', 'sot_token': '<|turn>', 'stc_token': '<|tool_call>', 'std_token': '<|tool>', 'str_token': '<|tool_response>', 'think_token': '<|think|>'}
Sep token is 105


In [16]:
def tokenize_function(batch):
    # Tokenize the processed input
    model_inputs = tokenizer(
        batch["Processed_Input"],
        truncation=True,
        max_length=1542,
        padding='max_length'
    )

    labels = []

    for input_ids in model_inputs["input_ids"]:
        label = list(input_ids)
        try:
            # Find the index of the first separation token
            indices = [i for i, val in enumerate(label) if val == sep_token_id]
            turn_model_idx = indices[-1]
            # Set all tokens up to the first separation to -100 (loss function ignore)
            for i in range(turn_model_idx + 1):
                label[i] = -100
        except ValueError as e:
            print(e)

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs


In [17]:
tokenized_train = train_set.map(tokenize_function, batched=True, remove_columns=train_set.column_names)
tokenized_test = test_set.map(tokenize_function, batched=True, remove_columns=test_set.column_names)

Map:   0%|          | 0/59651 [00:00<?, ? examples/s]

Map:   0%|          | 0/14913 [00:00<?, ? examples/s]

**Fine-tuning**

In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="summarization",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [30]:
model = model.to("cuda")

In [31]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [32]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 24.09 GiB. GPU 0 has a total capacity of 79.25 GiB of which 4.51 GiB is free. Including non-PyTorch memory, this process has 74.72 GiB memory in use. Of the allocated memory 65.51 GiB is allocated by PyTorch, and 8.71 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)